## Setup

Run the cells below to install dependencies and configure the language subset you want to fine-tune on.

In [1]:
!pip install -q seqeval evaluate #accelerate datasets evaluate seqeval transformers

In [2]:
import numpy as np
import torch
from datasets import get_dataset_config_names, load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
import evaluate

model_checkpoint = "castorini/afriberta_small"
available_languages = get_dataset_config_names("masakhane/masakhapos")
print(f"{len(available_languages)} languages available. Examples: {available_languages[:10]}")
language_code = "hau"  # Change this to another MasakhaNER2 language code when needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

20 languages available. Examples: ['bam', 'bbj', 'ewe', 'fon', 'hau', 'ibo', 'kin', 'lug', 'luo', 'mos']
Using device: cuda


In [3]:
from datasets import DatasetDict, concatenate_datasets

language_code = "multilingual"

raw_datasets_per_lang = [load_dataset("masakhane/masakhapos", lang) for lang in available_languages]

all_splits = set()
for ds in raw_datasets_per_lang:
    all_splits.update(ds.keys())

raw_datasets = DatasetDict(
    {
        split: concatenate_datasets([ds[split] for ds in raw_datasets_per_lang if split in ds])
        for split in sorted(all_splits)
    }
)
print(raw_datasets)

label_list = raw_datasets["train"].features["upos"].feature.names
num_labels = len(label_list)
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

split_names = list(raw_datasets.keys())
print(f"Available splits: {split_names}")
eval_split = "validation" if "validation" in raw_datasets else "test"

sample = raw_datasets["train"][0]
print("Sample tokens:", sample["tokens"])
print("Sample tags:", [label_list[tag] for tag in sample["upos"]])

DatasetDict({
    test: Dataset({
        features: ['id', 'tokens', 'upos'],
        num_rows: 12190
    })
    train: Dataset({
        features: ['id', 'tokens', 'upos'],
        num_rows: 15263
    })
    validation: Dataset({
        features: ['id', 'tokens', 'upos'],
        num_rows: 3041
    })
})
Available splits: ['test', 'train', 'validation']
Sample tokens: ['Muso', 'ŋana', ',', 'Afiriki', 'tilebinyanfan', "n'", 'a', 'cɛmancɛ', 'yanfan', 'ani', 'Magɛrɛbu', 'fana', "y'", 'u', 'bolo', 'don', 'min', 'kɔrɔ', ',', 'lagosira', ',', "k'", 'a', 'tɔgɔ', 'tiɲɛ', 'ani', "k'", 'a', 'lanɔgɔ', 'a', 'yɛrɛ', 'fasoden', 'ɲɔgɔnw', 'kɛ', ',', "k'", 'a', 'kɔrɔ', 'kɛ', 'ko', 'dɔw', 'dusu', 'tun', 'bɛna', 'kasi', 'ni', 'nin', 'ɲɔgɔnna', 'tɔgɔsɔrɔ', 'ye', 'Afirini', 'kunna', '.']
Sample tags: ['NOUN', 'ADJ', 'PUNCT', 'NOUN', 'NOUN', 'CCONJ', 'PRON', 'NOUN', 'ADV', 'CCONJ', 'NOUN', 'ADV', 'AUX', 'PRON', 'NOUN', 'VERB', 'PRON', 'ADP', 'PUNCT', 'VERB', 'PUNCT', 'PART', 'PRON', 'NOUN', 'VERB', 'CCON

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
label_all_tokens = False  # Set to True to propagate labels to all wordpieces

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )
    labels = []
    for batch_index, label_sequence in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_sequence[word_idx])
            else:
                label_ids.append(label_sequence[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
tokenized_datasets

/venv/main/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/15263 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


DatasetDict({
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 12190
    })
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15263
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3041
    })
})

In [5]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for prediction, label in zip(predictions, labels):
        filtered_preds = []
        filtered_labels = []
        for pred, lbl in zip(prediction, label):
            if lbl != -100:
                filtered_preds.append(label_list[pred])
                filtered_labels.append(label_list[lbl])
        true_predictions.append(filtered_preds)
        true_labels.append(filtered_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [6]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at castorini/afriberta_small and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForTokenClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(70006, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-3): 4 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, b

In [9]:
model_name = model_checkpoint.split("/")[-1]
run_name = f"{model_name}-{language_code}-ner"

training_args = TrainingArguments(
    output_dir=f"results/{run_name}",
    eval_strategy="epoch",
    # eval_steps=500,
    num_train_epochs=10,
    logging_steps=100,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    # num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    # save_strategy="no",
    # save_steps=500,
    # load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    gradient_accumulation_steps=1,
    # fp16=torch.cuda.is_available(),
    report_to="none",
)
training_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets[eval_split],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer

In [11]:
train_result = trainer.train()
train_result

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.716500,0.661073,0.767290,0.750361,0.758731,0.807918
2,0.478300,0.502341,0.817273,0.810736,0.813991,0.852380
3,0.388600,0.461249,0.833299,0.829200,0.831245,0.866405
4,0.320900,0.453605,0.840101,0.838755,0.839428,0.873101
5,0.281300,0.446386,0.845621,0.842432,0.844024,0.877243
6,0.243800,0.458965,0.845495,0.844286,0.844890,0.877481
7,0.215200,0.466358,0.847869,0.846816,0.847342,0.878955
8,0.204100,0.475923,0.847297,0.845779,0.846537,0.878745
9,0.180900,0.481670,0.849479,0.848011,0.848744,0.880486
10,0.173200,0.484451,0.849769,0.848608,0.849188,0.880640


/venv/main/lib/python3.13/site-packages/transformers/configuration_utils.py:461: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'max_length': 512}
  warnings.warn(
/venv/main/lib/python3.13/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/venv/main/lib/python3.13/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: AUX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/venv/main/lib/python3.13/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to b

TrainOutput(global_step=9540, training_loss=0.3868871660852332, metrics={'train_runtime': 241.7458, 'train_samples_per_second': 631.366, 'train_steps_per_second': 39.463, 'total_flos': 2983728916396404.0, 'train_loss': 0.3868871660852332, 'epoch': 10.0})

In [12]:
metrics = trainer.evaluate(tokenized_datasets["test"])
metrics

{'eval_loss': 0.5480648875236511,
 'eval_precision': 0.8370439363046855,
 'eval_recall': 0.8386850122927196,
 'eval_f1': 0.8378636707288887,
 'eval_accuracy': 0.8719295197052895,
 'eval_runtime': 10.9506,
 'eval_samples_per_second': 1113.182,
 'eval_steps_per_second': 69.585,
 'epoch': 10.0}

In [13]:
per_language_accuracy = {}
per_language_split = {}

for lang, ds_lang in zip(available_languages, raw_datasets_per_lang):
    splits = ds_lang.keys()
    if eval_split in splits:
        target_split = eval_split
    elif "test" in splits:
        target_split = "test"
    elif "validation" in splits:
        target_split = "validation"
    else:
        target_split = "train"

    tokenized_lang = ds_lang[target_split].map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=ds_lang[target_split].column_names,
    )
    metrics_lang = trainer.evaluate(tokenized_lang)
    accuracy = metrics_lang.get("eval_accuracy")
    if accuracy is not None:
        per_language_accuracy[lang] = accuracy
        per_language_split[lang] = target_split

average_accuracy = float(np.mean(list(per_language_accuracy.values()))) if per_language_accuracy else float("nan")

for lang, acc in per_language_accuracy.items():
    print(f"{lang} ({per_language_split[lang]}): {acc:.4f}")
print(f"\nAverage accuracy: {average_accuracy:.4f}")

/venv/main/lib/python3.13/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/venv/main/lib/python3.13/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


bam (validation): 0.8791
bbj (validation): 0.8126
ewe (validation): 0.8709
fon (validation): 0.8908
hau (validation): 0.9293
ibo (validation): 0.8142
kin (validation): 0.9699
lug (validation): 0.8854
luo (validation): 0.8510
mos (validation): 0.8810
nya (validation): 0.8033
pcm (validation): 0.8857
sna (validation): 0.8837
swa (validation): 0.9282
tsn (validation): 0.9086
twi (validation): 0.8219
wol (validation): 0.9031
xho (validation): 0.8035
yor (validation): 0.9341
zul (validation): 0.8389

Average accuracy: 0.8748


In [14]:
save_dir = f"results/{run_name}/final-model"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
save_dir

/venv/main/lib/python3.13/site-packages/transformers/configuration_utils.py:461: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'max_length': 512}
  warnings.warn(


'results/afriberta_small-multilingual-ner/final-model'

## Next Steps

- Adjust hyperparameters to better match dataset size and hardware.
- Enable logging integrations such as Weights & Biases by updating TrainingArguments.report_to.
- Push the fine-tuned weights to the Hugging Face Hub with trainer.push_to_hub() if you want to share the model.